# e-SNLI — Gemma3-27b-it SAE Layer 31 · Per-Feature Steering

Workflow:
1. Load e-SNLI dataset, Gemma-3-27b-it model, and a residual-stream SAE at a configurable layer (default 31).
2. Randomly sample one example and build the NLI prompt.
3. Generate the model response and verify the `<label>` tag matches the ground-truth label.
4. Collect residual-stream activations at the SAE layer and encode through the SAE.
5. Build a feature inventory: every feature that fires on at least one token (no Neuronpedia calls).
6. Iterate over **every active feature** one at a time; steer with coefficient `−max_activation`.
7. Classify each steered response as `correct` / `incorrect` / `invalid` and aggregate metrics.

**Optional — `FREEZE_ATTENTION=True`**: freezes self_attn outputs at layers `LAYER+1…end` during steering,
creating a linear mapping between the SAE delta and output logits for cleaner mechanistic isolation.

## Imports

In [1]:
import re
import sys
import os

import torch
import pandas as pd

sys.path.insert(0, os.path.dirname(os.getcwd()))

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

from src.configs import DatasetConfig, InferenceConfig, PromptStyle, SAEConfig
from src.dataset.esnli import ESNLI_Dataset
from src.SAE import JumpReLUSAE

## Configuration

In [2]:
LAYER = 31
# FREEZE_ATTENTION: if True, freeze self_attn outputs at layers LAYER+1..end
# during steering, so only the residual-stream perturbation from SAE features
# propagates — not downstream attention re-routing.
# Can be overridden via notebook parameter (papermill) or CLI: --freeze
FREEZE_ATTENTION = False   # default: standard autoregressive steering

sae_config = SAEConfig(
    repo_id="google/gemma-scope-2-27b-it",
    sae_type="resid_post",
    layer=LAYER,
    width="65k",
    l0="medium",
)
inference_config = InferenceConfig(max_new_tokens=512)
dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

# Parse --freeze flag when run as a script / via papermill param injection
import argparse
if "ipykernel" not in sys.modules:
    parser = argparse.ArgumentParser()
    parser.add_argument("--freeze", action="store_true")
    args, _ = parser.parse_known_args()
    FREEZE_ATTENTION = args.freeze

print(f"Model:            google/gemma-3-27b-it")
print(f"SAE layer:        {LAYER}")
print(f"SAE path:         {sae_config.sae_path}")
print(f"FREEZE_ATTENTION: {FREEZE_ATTENTION}")

Model:            google/gemma-3-27b-it
SAE layer:        31
SAE path:         resid_post/layer_31_width_65k_l0_medium/params.safetensors
FREEZE_ATTENTION: False


## HF Token

In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

## Load Dataset

In [4]:
esnli_dataset = ESNLI_Dataset(dataset_config)
prompted_data = esnli_dataset.build_prompts()
esnli_df = prompted_data.to_pandas()
print(f"Dataset size: {len(esnli_df)} examples")
esnli_df.head(3)

Data successfully loaded.
Dataset size: 9842 examples


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two women are embracing while holding to go pa...,The sisters are hugging goodbye while holding ...,1,The to go packages may not be from lunch.,"Just because two women are embracing, does not...",Two women do not have to be sisters. Embracin...,neutral,<start_of_turn>user Task: Determine the logica...
1,Two women are embracing while holding to go pa...,Two woman are holding packages.,0,Saying the two women are holding packages is a...,Sentence 1 states that two women are holding t...,Women can embrace while they are holding packa...,entailment,<start_of_turn>user Task: Determine the logica...
2,Two women are embracing while holding to go pa...,The men are fighting outside a deli.,2,In the first sentence there is an action of af...,Women are different than men and embracing is ...,First sentence features two women and the seco...,contradiction,<start_of_turn>user Task: Determine the logica...


## Load Model + SAE

In [5]:
device    = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
model     = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-27b-it", device_map=device, dtype=torch.bfloat16
)
model.eval()
print("Model loaded.")

sae = JumpReLUSAE.from_pretrained(sae_config, device=device)
sae.eval()
d_model = sae.w_dec.shape[1]
d_sae   = sae.w_dec.shape[0]
print(f"SAE loaded. d_model={d_model}, d_sae={d_sae}")

num_layers = len(model.model.language_model.layers)
print(f"num_layers={num_layers}")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Model loaded.
Load SAE resid_post/layer_31_width_65k_l0_medium/params.safetensors from google/gemma-scope-2-27b-it
SAE loaded. d_model=5376, d_sae=65536
num_layers=62


## Helper Functions

In [6]:
def generate_response(prompt: str):
    """Tokenize *prompt*, run greedy generation, return (full_text, full_ids, prompt_len)."""
    inputs     = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=inference_config.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    return text, output_ids, prompt_len


def collect_residual_activations(full_ids: torch.Tensor, layer_idx: int) -> torch.Tensor:
    """Hook the output of *layer_idx* to capture the residual stream.

    Returns a tensor of shape (n_tokens, d_model) on CPU.
    """
    cache  = {}
    layer  = model.model.language_model.layers[layer_idx]
    handle = layer.register_forward_hook(
        lambda _m, _i, out: cache.__setitem__(
            "resid",
            (out[0] if isinstance(out, tuple) else out).detach().squeeze(0),
        )
    )
    try:
        with torch.no_grad():
            model(input_ids=full_ids, use_cache=False)
    finally:
        handle.remove()
    return cache["resid"]   # (n_tokens, d_model)


def run_steered_generation(
    prompt_ids: torch.Tensor,
    layer_idx: int,
    steering_delta: torch.Tensor,
) -> torch.Tensor:
    """Autoregressive generation with a residual-stream steering hook.

    Injects *steering_delta* (shape ``(d_model,)``) at every decoding step by
    hooking the output of the residual stream at *layer_idx*.

    Returns the full output token-ID tensor (1, seq_len).
    """
    layer = model.model.language_model.layers[layer_idx]

    def _hook(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        hidden = hidden + steering_delta.to(dtype=hidden.dtype, device=hidden.device)
        if isinstance(out, tuple):
            return (hidden,) + out[1:]
        return hidden

    handle = layer.register_forward_hook(_hook)
    try:
        with torch.no_grad():
            steered_ids = model.generate(
                input_ids=prompt_ids,
                attention_mask=torch.ones_like(prompt_ids),
                max_new_tokens=inference_config.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    return steered_ids


def decode_response(ids: torch.Tensor) -> str:
    """Decode *ids* and return the model-turn text only."""
    full = tokenizer.decode(ids[0], skip_special_tokens=True)
    return full.split("<start_of_turn>model")[-1].strip()


def capture_attn_cache(full_ids: torch.Tensor, layer_idx: int) -> dict:
    """Capture self_attn outputs for layers layer_idx+1..num_layers-1.

    Returns attn_cache: dict[int -> Tensor(1, n_tokens, d_model)].
    """
    attn_cache = {}
    handles    = []

    def make_capture(L):
        def hook(module, inp, out):
            attn_cache[L] = out[0].detach().clone()
        return hook

    for L in range(layer_idx + 1, num_layers):
        handles.append(
            model.model.language_model.layers[L].self_attn.register_forward_hook(make_capture(L))
        )
    try:
        with torch.no_grad():
            model(input_ids=full_ids, use_cache=False)
    finally:
        for h in handles:
            h.remove()
    return attn_cache


def run_steered_generation_frozen_attn(
    full_ids: torch.Tensor,
    layer_idx: int,
    steering_delta: torch.Tensor,
    attn_cache: dict,
) -> str:
    """Single forward pass on full_ids with residual steering + frozen downstream attention.

    Returns greedily decoded text of the response portion (teacher-forced).
    """
    handles  = []
    n_prompt = prompt_ids.shape[1]

    def _steer(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        hidden = hidden + steering_delta.to(dtype=hidden.dtype, device=hidden.device)
        if isinstance(out, tuple):
            return (hidden,) + out[1:]
        return hidden

    handles.append(
        model.model.language_model.layers[layer_idx].register_forward_hook(_steer)
    )

    def make_freeze(cached):
        def hook(module, inp, out):
            return (cached.to(dtype=out[0].dtype),) + out[1:]
        return hook

    for L in range(layer_idx + 1, num_layers):
        handles.append(
            model.model.language_model.layers[L].self_attn.register_forward_hook(
                make_freeze(attn_cache[L])
            )
        )

    try:
        with torch.no_grad():
            logits = model(input_ids=full_ids, use_cache=False).logits.squeeze(0)
        # Teacher-forced greedy decode of response tokens.
        # logits[i] is the prediction for position i+1, so
        # logits[n_prompt-1 : -1] predicts the response token positions.
        pred_ids = logits[n_prompt - 1 : -1].argmax(dim=-1)
        return tokenizer.decode(pred_ids, skip_special_tokens=True)
    finally:
        for h in handles:
            h.remove()


print("Helper functions defined.")

Helper functions defined.


## Sample + Baseline Verification

Randomly pick one example and verify the model produces the correct label.
Re-sample up to `MAX_RETRIES` times if the model answers incorrectly.

In [7]:
SEED        = 42
MAX_RETRIES = 20

sample_row  = None
sample_text = None
full_ids    = None
prompt_ids  = None
prompt_len  = None
correct_label = None

candidate_df = esnli_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

for attempt, (_, row) in enumerate(candidate_df.iterrows()):
    if attempt >= MAX_RETRIES:
        raise RuntimeError(f"Could not find a correctly-answered sample in {MAX_RETRIES} tries.")

    prompt     = row["prompt"]
    gold_label = row["gold_label"].lower()

    text, out_ids, p_len = generate_response(prompt)
    parsed = esnli_dataset.parse_model_answer(text)
    predicted = parsed.label if parsed is not None else None

    print(f"  Attempt {attempt + 1}: Predicted={predicted!r}  Gold={gold_label!r}")

    if predicted == gold_label:
        sample_row    = row
        sample_text   = text
        full_ids      = out_ids
        prompt_len    = p_len
        prompt_ids    = out_ids[:, :p_len]
        correct_label = gold_label
        print("Found a correctly-answered sample.")
        break

print(f"\nSelected sample index: {sample_row.name}")
print(f"Premise:    {sample_row['premise']}")
print(f"Hypothesis: {sample_row['hypothesis']}")
print(f"Gold label: {sample_row['gold_label']}")
print(f"Prompt tokens: {prompt_len}  |  Total tokens: {full_ids.shape[1]}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Attempt 1: Predicted='neutral'  Gold='contradiction'
  Attempt 2: Predicted='neutral'  Gold='contradiction'
  Attempt 3: Predicted='contradiction'  Gold='contradiction'
Found a correctly-answered sample.

Selected sample index: 2
Premise:    A lady sitting on a bench that is against a building and under a poster of a man in a uniform waving.
Hypothesis: Nobody is sitting
Gold label: contradiction
Prompt tokens: 109  |  Total tokens: 186


## Collect Residual Stream Activations + SAE Encode

Hook the layer output (post-MLP residual stream) at `LAYER`. Encode with the SAE.
If `FREEZE_ATTENTION=True`, also capture the baseline attention cache for frozen-attn steering.

In [8]:
resid_acts = collect_residual_activations(full_ids, LAYER)   # (n_tokens, d_model)
print(f"Residual activations shape: {resid_acts.shape}")

with torch.no_grad():
    sae_acts = sae.encode(resid_acts.float())   # (n_tokens, d_sae)

print(f"SAE activations shape: {sae_acts.shape}")
print(f"Mean L0 over sequence: {(sae_acts > 0).float().sum(dim=-1).mean():.1f}")

if FREEZE_ATTENTION:
    attn_cache = capture_attn_cache(full_ids, LAYER)
    print(f"Captured attention cache for {len(attn_cache)} layers")
else:
    attn_cache = None
    print("Attention freezing disabled (FREEZE_ATTENTION=False).")

Residual activations shape: torch.Size([186, 5376])
SAE activations shape: torch.Size([186, 65536])
Mean L0 over sequence: 57.2
Attention freezing disabled (FREEZE_ATTENTION=False).


## Build Feature Inventory

Collect every feature that fires on at least one token. No Neuronpedia calls — just raw activations.

In [9]:
active_features = []
for fi in range(sae_acts.shape[1]):
    acts = sae_acts[:, fi]
    if (acts > 0).any():
        active_features.append({"feature_idx": fi, "max_activation": acts.max().item()})

feature_df = pd.DataFrame(active_features)
print(f"Total active features: {len(feature_df)}")
feature_df.head(10)

Total active features: 3812


,feature_idx,max_activation
0,0,3035.127197
1,1,1194.930664
2,6,20448.833984
3,8,1900.177734
4,12,16403.835938
5,14,214.954590
6,20,932.897461
7,25,1116.075806
8,26,586.263062
9,29,408.593597


## Per-Feature Steering Loop

For each active feature, steer with coefficient `−max_activation` and classify the result.

- `FREEZE_ATTENTION=False`: standard autoregressive generation via `model.generate()` (same as source notebook).
- `FREEZE_ATTENTION=True`: single forward pass on (prompt + baseline response) with steering at `LAYER`
  and frozen self_attn at `LAYER+1…num_layers-1`; teacher-forced greedy decode.

In [10]:
steering_results = []

for i, row in feature_df.iterrows():
    fi    = int(row["feature_idx"])
    coeff = -row["max_activation"]
    delta = (coeff * sae.w_dec[fi].float()).to(device)

    if FREEZE_ATTENTION:
        steered_text = run_steered_generation_frozen_attn(full_ids, LAYER, delta, attn_cache)
    else:
        steered_ids  = run_steered_generation(prompt_ids, LAYER, delta)
        steered_text = decode_response(steered_ids)

    parsed = esnli_dataset.parse_model_answer(steered_text)
    pred   = parsed.label if parsed is not None else None

    if pred is None:
        status = "invalid"
    elif pred == correct_label:
        status = "correct"
    else:
        status = "incorrect"

    steering_results.append({
        "feature_idx":     fi,
        "max_activation":  row["max_activation"],
        "predicted_label": pred,
        "status":          status,
    })

    if (i + 1) % 100 == 0:
        print(f"  [{i + 1}/{len(feature_df)}] feature {fi}: status={status}, pred={pred!r}")

print(f"\nSteering loop complete. Processed {len(steering_results)} features.")

  [100/3812] feature 301: status=correct, pred='contradiction'
  [200/3812] feature 571: status=correct, pred='contradiction'
  [300/3812] feature 875: status=incorrect, pred='InvalidFormat'
  [400/3812] feature 1168: status=correct, pred='contradiction'
  [500/3812] feature 1458: status=correct, pred='contradiction'
  [600/3812] feature 1753: status=correct, pred='contradiction'
  [700/3812] feature 2026: status=correct, pred='contradiction'
  [800/3812] feature 2317: status=correct, pred='contradiction'
  [900/3812] feature 2664: status=correct, pred='contradiction'
  [1000/3812] feature 3114: status=correct, pred='contradiction'
  [1100/3812] feature 3557: status=correct, pred='contradiction'
  [1200/3812] feature 4161: status=correct, pred='contradiction'


KeyboardInterrupt: 

## Aggregate Metrics

In [ ]:
results_df = pd.DataFrame(steering_results)
total       = len(results_df)
n_correct   = (results_df["status"] == "correct").sum()
n_incorrect = (results_df["status"] == "incorrect").sum()
n_invalid   = (results_df["status"] == "invalid").sum()
n_changed   = n_incorrect + n_invalid

assert n_correct + n_incorrect + n_invalid == total, "Counts don't sum to total!"

print(f"FREEZE_ATTENTION = {FREEZE_ATTENTION}")
print(f"Gold label: '{correct_label}'")
print()
print(f"Total active features steered : {total}")
print(f"Answer unchanged (correct)    : {n_correct}  ({100*n_correct/total:.1f}%)")
print(f"Answer changed (incorrect)    : {n_incorrect} ({100*n_incorrect/total:.1f}%)")
print(f"Answer invalid / unparseable  : {n_invalid}  ({100*n_invalid/total:.1f}%)")
print(f"Answer changed (any)          : {n_changed}  ({100*n_changed/total:.1f}%)")
print(f"\nPredicted label distribution (all steerings):")
print(results_df["predicted_label"].value_counts(dropna=False))
print(f"\nFeatures that changed the answer (sorted by max_activation desc):")
changed = results_df[results_df["status"] != "correct"].sort_values("max_activation", ascending=False)
print(changed[["feature_idx", "max_activation", "predicted_label", "status"]].to_string())

FREEZE_ATTENTION = False
Gold label: 'contradiction'

Total active features steered : 1261
Answer unchanged (correct)    : 1139  (90.3%)
Answer changed (incorrect)    : 122 (9.7%)
Answer invalid / unparseable  : 0  (0.0%)
Answer changed (any)          : 122  (9.7%)

Predicted label distribution (all steerings):
predicted_label
contradiction    1139
InvalidFormat     122
Name: count, dtype: int64

Features that changed the answer (sorted by max_activation desc):
     feature_idx  max_activation predicted_label     status
509         1482    75778.125000   InvalidFormat  incorrect
560         1640    71047.187500   InvalidFormat  incorrect
264          755    66777.171875   InvalidFormat  incorrect
410         1212    52900.589844   InvalidFormat  incorrect
228          662    49772.367188   InvalidFormat  incorrect
481         1403    46316.101562   InvalidFormat  incorrect
255          736    46109.027344   InvalidFormat  incorrect
280          802    43952.554688   InvalidFormat  inco

: 